# Causal Impact Analysis: Jaguar Release

## Overview

**Objective:** Measure the incremental lift of Jaguar release on KPIs using synthetic control methodology.

**Approach:**
1. Pull daily KPI data from coredw (pre and post release)
2. Use CausalImpact to model counterfactual (what KPIs would have been without intervention)
3. Covariates: spend, impressions, uniques help predict expected baseline
4. Primary metric: IVR (with all other KPIs available)

**Methodology:** Google's CausalImpact uses Bayesian structural time series to estimate the causal effect of an intervention by constructing a synthetic control from covariates.

## Data Sources

| Source | Tables | Purpose |
|--------|--------|----------|
| coredw (Greenplum) | `summarydata.sum_by_campaign_group_by_day`, `fpa.advertiser_verticals`, etc. | Daily KPIs by vertical |

---
## 1. Setup & Imports

In [ ]:
# =============================================================================
# INSTALL DEPENDENCIES (run once per cluster)
# =============================================================================

%pip install pycausalimpact --quiet

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import json
from datetime import date, timedelta
from typing import Dict, List, Optional

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import col

import google.auth.transport.requests as g_request
from google.auth import compute_engine
import requests

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from causalimpact import CausalImpact

# Initialize Spark session
spark = SparkSession.builder.appName("causal_impact_jaguar").getOrCreate()

print("Setup complete.")

---
## 2. Configuration Parameters

In [ ]:
# =============================================================================
# ANALYSIS PARAMETERS
# =============================================================================

# Release date (intervention date - excluded from analysis)
RELEASE_DATE = date(2025, 11, 20)

# Vertical filter: list of vertical IDs, or None for all verticals
VERTICAL_IDS: Optional[List[int]] = [113002]  # Set to None for all verticals

# Primary metric to analyze (others available as covariates)
PRIMARY_METRIC = "ivr"

# =============================================================================
# DERIVED DATES
# =============================================================================

# Calculate period length (7-day increments)
days_since_release = (date.today() - timedelta(days=1) - RELEASE_DATE).days

if days_since_release < 7:
    PERIOD_DAYS = days_since_release
else:
    PERIOD_DAYS = (days_since_release // 7) * 7

# Pre period: PERIOD_DAYS before release
PRE_START = RELEASE_DATE - timedelta(days=PERIOD_DAYS)
PRE_END = RELEASE_DATE - timedelta(days=1)

# Post period: PERIOD_DAYS after release
POST_START = RELEASE_DATE + timedelta(days=1)
POST_END = RELEASE_DATE + timedelta(days=PERIOD_DAYS)

print(f"Release Date:      {RELEASE_DATE}")
print(f"Period Length:     {PERIOD_DAYS} days")
print(f"Pre Period:        {PRE_START} to {PRE_END}")
print(f"Post Period:       {POST_START} to {POST_END}")
print(f"Vertical IDs:      {VERTICAL_IDS if VERTICAL_IDS else 'All verticals'}")
print(f"Primary Metric:    {PRIMARY_METRIC.upper()}")

---
## 3. Helper Functions

In [ ]:
# =============================================================================
# VAULT & DATABASE HELPERS
# =============================================================================

def token_for_url(url: str) -> str:
    """
    Get GCP identity token for the specified URL.
    Used for Vault authentication via workload identity.
    """
    request = g_request.Request()
    credentials = compute_engine.IDTokenCredentials(
        request=request,
        target_audience=url,
        use_metadata_identity_endpoint=True,
    )
    credentials.refresh(request)
    return credentials.token


def get_secret(secret_name: str) -> Dict:
    """
    Retrieve secret from Vault using GCP workload identity authentication.
    """
    vault_address = "https://vault.prod.in.mountain.com"
    role = "gcp-workloads"
    path = "shared/global/ti"

    jwt = token_for_url(f"{vault_address}/vault/gcp-workloads")

    auth_resp = requests.post(
        f"{vault_address}/v1/auth/gcp/login",
        headers={"Content-Type": "application/json"},
        data=json.dumps({"role": role, "jwt": jwt}),
    )
    auth_resp.raise_for_status()
    vault_token = auth_resp.json()["auth"]["client_token"]

    secret_resp = requests.get(
        f"{vault_address}/v1/secret/data/{path}/{secret_name}",
        headers={"X-Vault-Token": vault_token},
    )
    secret_resp.raise_for_status()

    return secret_resp.json().get("data", {}).get("data")


def load_postgres_query(query: str, session: SparkSession) -> DataFrame:
    """
    Execute a query against coredw (Greenplum) and return results as Spark DataFrame.
    """
    secrets = get_secret("coredw")

    return (
        session.read
        .format("jdbc")
        .option("url", f"jdbc:postgresql://{secrets['hostname']}:{secrets['port']}/{secrets['database']}")
        .option("dbtable", query)
        .option("user", secrets["username"])
        .option("password", secrets["password"])
        .option("driver", "org.postgresql.Driver")
        .load()
    )


print("Helper functions loaded.")

---
## 4. Build Daily KPI Query

In [ ]:
# =============================================================================
# BUILD SQL QUERY FOR DAILY KPIs
# =============================================================================

# Build vertical filter clause
if VERTICAL_IDS:
    vertical_filter = f"and av.vertical_id in ({','.join(map(str, VERTICAL_IDS))})"
else:
    vertical_filter = "-- all verticals (no filter)"

DAILY_KPI_QUERY = f"""
(
    /* ------------------------------------------------------------------------
       Step 1: Get advertisers in target vertical(s)
       ------------------------------------------------------------------------ */
    with advertisers_in_vertical as (
        select distinct
            av.advertiser_id
          , av.vertical_id
          , av.vertical_name
        from fpa.advertiser_verticals av
        where 1 = 1
            and av.type = 1
            {vertical_filter}
    )

    /* ------------------------------------------------------------------------
       Step 2: Get Funnel 1 campaign groups
       ------------------------------------------------------------------------ */
    , qualifying_campaigns as (
        select distinct
            a.advertiser_id
          , a.vertical_id
          , a.vertical_name
          , cg.campaign_group_id
        from advertisers_in_vertical a
        inner join public.campaign_groups cg
            on cg.advertiser_id = a.advertiser_id
        inner join campaign_groups_raw cgr
            on cgr.campaign_group_id = cg.campaign_group_id
        where 1 = 1
            and cgr.objective_id = 1
    )

    /* ------------------------------------------------------------------------
       Step 3: Identify last-touch attribution advertisers
       ------------------------------------------------------------------------ */
    , last_touch_advertisers as (
        select distinct advertiser_id
        from r2.advertiser_settings
        where reporting_style = 'last_touch'
    )

    /* ------------------------------------------------------------------------
       Step 4: Aggregate daily KPIs by vertical
       ------------------------------------------------------------------------ */
    select
        qc.vertical_id
      , qc.vertical_name
      , d.day

        -- Activity counts
      , count(distinct d.campaign_group_id) as active_campaigns
      , count(distinct qc.advertiser_id) as active_advertisers

        -- Volume metrics
      , sum(d.impressions) as impressions
      , sum(d.media_spend + d.data_spend + d.platform_spend) as spend

        -- Conversions (attribution-aware)
      , sum(
            case
                when lt.advertiser_id is null
                    then d.click_conversions + d.view_conversions + coalesce(d.competing_view_conversions, 0)
                else d.click_conversions + d.view_conversions
            end
        ) as conversions

        -- Order value (attribution-aware)
      , sum(
            case
                when lt.advertiser_id is null
                    then d.click_order_value + d.view_order_value + coalesce(d.competing_view_order_value, 0)
                else d.click_order_value + d.view_order_value
            end
        ) as order_value

        -- Verified visits (attribution-aware)
      , sum(
            case
                when lt.advertiser_id is null
                    then d.clicks + d.views + coalesce(d.competing_views, 0)
                else d.clicks + d.views
            end
        ) as vv

        -- Uniques
      , sum(d.uniques) as uniques

    from qualifying_campaigns qc
    inner join summarydata.sum_by_campaign_group_by_day d
        on d.campaign_group_id = qc.campaign_group_id
    left join last_touch_advertisers lt
        on lt.advertiser_id = qc.advertiser_id
    where 1 = 1
        and d.day >= '{PRE_START}'::date
        and d.day <= '{POST_END}'::date
        and d.day <> '{RELEASE_DATE}'::date  -- exclude release date
        and d.impressions > 0
    group by 1, 2, 3
    order by 1, 3
) as daily_kpis
"""

print("Query built successfully.")
print(f"\nVertical filter: {vertical_filter}")
print(f"Date range: {PRE_START} to {POST_END} (excluding {RELEASE_DATE})")

---
## 5. Load Data from coredw

In [ ]:
# =============================================================================
# LOAD DAILY KPI DATA
# =============================================================================

daily_kpis_df = load_postgres_query(DAILY_KPI_QUERY, spark)

# Convert to pandas for CausalImpact
daily_kpis_pd = daily_kpis_df.toPandas()

# Convert day to datetime
daily_kpis_pd["day"] = pd.to_datetime(daily_kpis_pd["day"])

print(f"Loaded {len(daily_kpis_pd):,} daily records")
print(f"Verticals: {daily_kpis_pd['vertical_name'].nunique()}")
print(f"Date range: {daily_kpis_pd['day'].min().date()} to {daily_kpis_pd['day'].max().date()}")

daily_kpis_pd.head()

In [ ]:
# =============================================================================
# CALCULATE EFFICIENCY METRICS
# =============================================================================

# Add calculated KPI columns
daily_kpis_pd["ivr"] = daily_kpis_pd["vv"] / daily_kpis_pd["impressions"].replace(0, np.nan)
daily_kpis_pd["cvr"] = daily_kpis_pd["conversions"] / daily_kpis_pd["vv"].replace(0, np.nan)
daily_kpis_pd["vvr"] = daily_kpis_pd["vv"] / daily_kpis_pd["uniques"].replace(0, np.nan)
daily_kpis_pd["cpa"] = daily_kpis_pd["spend"] / daily_kpis_pd["conversions"].replace(0, np.nan)
daily_kpis_pd["cpv"] = daily_kpis_pd["spend"] / daily_kpis_pd["vv"].replace(0, np.nan)
daily_kpis_pd["roas"] = daily_kpis_pd["order_value"] / daily_kpis_pd["spend"].replace(0, np.nan)
daily_kpis_pd["aov"] = daily_kpis_pd["order_value"] / daily_kpis_pd["conversions"].replace(0, np.nan)

print("Efficiency metrics calculated.")
print(f"\nAvailable metrics: {['ivr', 'cvr', 'vvr', 'cpa', 'cpv', 'roas', 'aov']}")
print(f"Available covariates: {['spend', 'impressions', 'uniques', 'vv', 'conversions']}")

In [ ]:
# =============================================================================
# VALIDATION: CHECK DATA COMPLETENESS
# =============================================================================

release_date_dt = pd.Timestamp(RELEASE_DATE)

validation = daily_kpis_pd.groupby("vertical_name").agg(
    pre_days=pd.NamedAgg(column="day", aggfunc=lambda x: (x < release_date_dt).sum()),
    post_days=pd.NamedAgg(column="day", aggfunc=lambda x: (x > release_date_dt).sum()),
    total_impressions=pd.NamedAgg(column="impressions", aggfunc="sum"),
    avg_ivr=pd.NamedAgg(column="ivr", aggfunc="mean"),
).reset_index()

print("Data completeness by vertical:")
display(validation)

---
## 6. Causal Impact Analysis Functions

In [ ]:
# =============================================================================
# CAUSAL IMPACT HELPER FUNCTIONS
# =============================================================================

def run_causal_impact(
    data: pd.DataFrame,
    metric: str,
    covariates: List[str],
    pre_period: tuple,
    post_period: tuple,
    vertical_name: str = "All"
) -> dict:
    """
    Run CausalImpact analysis on a single metric.
    
    Args:
        data: DataFrame with 'day' as index and metric + covariates as columns
        metric: Target metric column name (e.g., 'ivr')
        covariates: List of covariate column names (e.g., ['spend', 'impressions'])
        pre_period: Tuple of (start_date, end_date) for pre-intervention period
        post_period: Tuple of (start_date, end_date) for post-intervention period
        vertical_name: Name for labeling output
    
    Returns:
        Dictionary with results summary
    """
    # Prepare data: metric first, then covariates
    columns = [metric] + covariates
    ci_data = data[columns].copy()
    
    # Drop rows with NaN in the target metric
    ci_data = ci_data.dropna(subset=[metric])
    
    # Fill NaN in covariates with forward fill then backward fill
    ci_data[covariates] = ci_data[covariates].ffill().bfill()
    
    if len(ci_data) < 10:
        print(f"  ⚠️  Insufficient data for {vertical_name} ({len(ci_data)} rows)")
        return None
    
    try:
        # Run CausalImpact
        ci = CausalImpact(ci_data, pre_period, post_period)
        
        # Extract summary statistics
        summary = ci.summary_data
        
        result = {
            "vertical": vertical_name,
            "metric": metric,
            "pre_period": f"{pre_period[0].date()} to {pre_period[1].date()}",
            "post_period": f"{post_period[0].date()} to {post_period[1].date()}",
            "actual_avg": summary.loc["average", "actual"],
            "predicted_avg": summary.loc["average", "predicted"],
            "absolute_effect": summary.loc["average", "abs_effect"],
            "relative_effect": summary.loc["average", "rel_effect"],
            "ci_lower": summary.loc["average", "abs_effect_lower"],
            "ci_upper": summary.loc["average", "abs_effect_upper"],
            "p_value": ci.p_value,
            "significant": ci.p_value < 0.05,
            "ci_object": ci  # Store for plotting
        }
        
        return result
        
    except Exception as e:
        print(f"  ❌ Error for {vertical_name}: {str(e)}")
        return None


def plot_causal_impact(result: dict, figsize=(14, 10)):
    """
    Plot CausalImpact results.
    """
    if result is None or "ci_object" not in result:
        print("No results to plot.")
        return
    
    ci = result["ci_object"]
    
    fig = ci.plot(figsize=figsize)
    fig.suptitle(
        f"Causal Impact: {result['metric'].upper()} - {result['vertical']}\n"
        f"Relative Effect: {result['relative_effect']:.2%} (p={result['p_value']:.4f})",
        fontsize=14,
        fontweight="bold"
    )
    plt.tight_layout()
    plt.show()


def summarize_results(results: List[dict]) -> pd.DataFrame:
    """
    Create summary DataFrame from list of results.
    """
    # Filter out None results and remove ci_object for display
    clean_results = []
    for r in results:
        if r is not None:
            r_copy = {k: v for k, v in r.items() if k != "ci_object"}
            clean_results.append(r_copy)
    
    if not clean_results:
        return pd.DataFrame()
    
    df = pd.DataFrame(clean_results)
    
    # Format for display
    df["relative_effect_pct"] = df["relative_effect"].apply(lambda x: f"{x:.2%}")
    df["p_value_fmt"] = df["p_value"].apply(lambda x: f"{x:.4f}")
    df["significance"] = df["significant"].apply(lambda x: "✓ Significant" if x else "Not significant")
    
    return df


print("Causal Impact functions loaded.")

---
## 7. Run Causal Impact Analysis

In [ ]:
# =============================================================================
# CONFIGURATION FOR ANALYSIS
# =============================================================================

# Define periods as timestamps
pre_period = (pd.Timestamp(PRE_START), pd.Timestamp(PRE_END))
post_period = (pd.Timestamp(POST_START), pd.Timestamp(POST_END))

# Covariates to use for synthetic control
# These help predict what the metric "would have been" without intervention
COVARIATES = ["spend", "impressions", "uniques"]

# Metrics to analyze
METRICS_TO_ANALYZE = ["ivr"]  # Add more: ["ivr", "cvr", "roas", "cpa"]

print(f"Pre-period:  {pre_period[0].date()} to {pre_period[1].date()}")
print(f"Post-period: {post_period[0].date()} to {post_period[1].date()}")
print(f"Covariates:  {COVARIATES}")
print(f"Metrics:     {METRICS_TO_ANALYZE}")

In [ ]:
# =============================================================================
# RUN ANALYSIS PER VERTICAL
# =============================================================================

all_results = []

verticals = daily_kpis_pd["vertical_name"].unique()
print(f"Analyzing {len(verticals)} vertical(s)...\n")

for vertical in verticals:
    print(f"📊 Processing: {vertical}")
    
    # Filter data for this vertical
    vertical_data = daily_kpis_pd[daily_kpis_pd["vertical_name"] == vertical].copy()
    vertical_data = vertical_data.set_index("day").sort_index()
    
    for metric in METRICS_TO_ANALYZE:
        print(f"   → Metric: {metric.upper()}")
        
        result = run_causal_impact(
            data=vertical_data,
            metric=metric,
            covariates=COVARIATES,
            pre_period=pre_period,
            post_period=post_period,
            vertical_name=vertical
        )
        
        if result:
            all_results.append(result)
            print(f"      Effect: {result['relative_effect']:.2%} (p={result['p_value']:.4f})")
    
    print()

print(f"\n✅ Completed {len(all_results)} analyses.")

---
## 8. Results Summary

In [ ]:
# =============================================================================
# SUMMARY TABLE
# =============================================================================

summary_df = summarize_results(all_results)

if not summary_df.empty:
    display_cols = [
        "vertical", 
        "metric", 
        "actual_avg", 
        "predicted_avg", 
        "relative_effect_pct", 
        "p_value_fmt", 
        "significance"
    ]
    
    print("\n" + "="*80)
    print("CAUSAL IMPACT SUMMARY")
    print("="*80)
    display(summary_df[display_cols])
else:
    print("No results to display.")

In [ ]:
# =============================================================================
# INTERPRETATION
# =============================================================================

if all_results:
    for result in all_results:
        if result:
            print(f"\n{'='*80}")
            print(f"INTERPRETATION: {result['vertical']} - {result['metric'].upper()}")
            print(f"{'='*80}")
            print(result["ci_object"].summary())

---
## 9. Visualizations

In [ ]:
# =============================================================================
# CAUSAL IMPACT PLOTS
# =============================================================================

for result in all_results:
    if result:
        plot_causal_impact(result)

In [ ]:
# =============================================================================
# RAW TIME SERIES PLOT (PRE/POST COMPARISON)
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 6))

release_date_dt = pd.Timestamp(RELEASE_DATE)

for vertical in daily_kpis_pd["vertical_name"].unique():
    v_data = daily_kpis_pd[daily_kpis_pd["vertical_name"] == vertical].sort_values("day")
    ax.plot(v_data["day"], v_data[PRIMARY_METRIC], marker="o", markersize=4, label=vertical)

# Add vertical line at release date
ax.axvline(x=release_date_dt, color="red", linestyle="--", linewidth=2, label="Release Date")

# Shade pre/post regions
ax.axvspan(pd.Timestamp(PRE_START), release_date_dt, alpha=0.1, color="blue", label="Pre-period")
ax.axvspan(release_date_dt, pd.Timestamp(POST_END), alpha=0.1, color="green", label="Post-period")

ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel(PRIMARY_METRIC.upper(), fontsize=12)
ax.set_title(f"Daily {PRIMARY_METRIC.upper()} - Pre/Post Comparison", fontsize=14)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 10. Export Results (Optional)

In [ ]:
# =============================================================================
# EXPORT TO CSV (uncomment to use)
# =============================================================================

# # Export summary
# if not summary_df.empty:
#     export_df = summary_df.drop(columns=["ci_object"], errors="ignore")
#     export_df.to_csv("/dbfs/tmp/causal_impact_summary.csv", index=False)
#     print("Summary exported to /dbfs/tmp/causal_impact_summary.csv")

# # Export daily data
# daily_kpis_pd.to_csv("/dbfs/tmp/causal_impact_daily_data.csv", index=False)
# print("Daily data exported to /dbfs/tmp/causal_impact_daily_data.csv")

---
## 11. Run Additional Metrics (Optional)

In [ ]:
# =============================================================================
# ANALYZE ALL KPIS (uncomment to run)
# =============================================================================

# ALL_METRICS = ["ivr", "cvr", "vvr", "cpa", "cpv", "roas", "aov"]

# all_kpi_results = []

# for vertical in daily_kpis_pd["vertical_name"].unique():
#     print(f"\n📊 Processing: {vertical}")
#     vertical_data = daily_kpis_pd[daily_kpis_pd["vertical_name"] == vertical].copy()
#     vertical_data = vertical_data.set_index("day").sort_index()
    
#     for metric in ALL_METRICS:
#         result = run_causal_impact(
#             data=vertical_data,
#             metric=metric,
#             covariates=COVARIATES,
#             pre_period=pre_period,
#             post_period=post_period,
#             vertical_name=vertical
#         )
#         if result:
#             all_kpi_results.append(result)

# full_summary = summarize_results(all_kpi_results)
# display(full_summary[["vertical", "metric", "relative_effect_pct", "p_value_fmt", "significance"]])